# W2/D1 — Alert Correlation Lab
**Goal:** Gom 20 alert thành cluster có nghĩa bằng 3-layer pipeline:
1. **Dedup** (fingerprint)
2. **Time-Window** (session grouping)
3. **Topology-Aware** (service graph distance)

---

In [17]:
# Import & Load Data
import json
import os
import sys
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import networkx as nx

from correlate import (
    load_alerts,
    build_graph,
    fingerprint,
    Deduper,
    session_groups,
    topology_group,
    correlate,
    build_summary,
)

DATASET_DIR = Path("dataset")
alerts = load_alerts(str(DATASET_DIR / "alerts_sample.jsonl"))
graph  = build_graph(str(DATASET_DIR / "services.json"))

print(f"Loaded {len(alerts)} alerts")
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print()
print("-- Sample alerts --")
for a in alerts[:3]:
    print(f"  {a['id']}  {a['ts']}  {a['service']:20s}  {a['metric']:35s}  {a['severity']}  val={a['value']}")

Loaded 20 alerts
Graph: 14 nodes, 17 edges

-- Sample alerts --
  a-0001  2026-06-12T09:42:01Z  payment-svc           db_connection_pool_used_ratio        warn  val=0.85
  a-0002  2026-06-12T09:42:18Z  payment-svc           db_connection_pool_used_ratio        crit  val=0.99
  a-0003  2026-06-12T09:42:22Z  payment-svc           latency_p99_ms                       crit  val=1840


In [18]:
# Layer 1 - Fingerprint Dedup
deduper = Deduper()
for a in alerts:
    deduper.push(a)

dedup_clusters = deduper.clusters()
print("Layer 1 - Dedup Results")
print("=" * 60)
print(f"Input alerts:        {len(alerts)}")
print(f"Unique fingerprints: {len(dedup_clusters)}")
print(f"Dedup ratio:         {1 - len(dedup_clusters)/len(alerts):.1%}")
print()

print(f"{'Fingerprint':55s} {'Count':>5}  Alert IDs")
print(f"{'-'*55} {'-'*5}  {'-'*20}")
for c in sorted(dedup_clusters, key=lambda x: -x['count']):
    print(f"{c['fingerprint']:55s} {c['count']:5d}  {c['alerts']}")

print()
dup_fps = [c for c in dedup_clusters if c['count'] > 1]
print(f"{len(dup_fps)} fingerprint(s) have duplicates (count > 1)")

Layer 1 - Dedup Results
Input alerts:        20
Unique fingerprints: 17
Dedup ratio:         15.0%

Fingerprint                                             Count  Alert IDs
------------------------------------------------------- -----  --------------------
payment-svc|latency_p99_ms|crit                             3  ['a-0003', 'a-0008', 'a-0015']
payment-svc|db_connection_pool_used_ratio|crit              2  ['a-0002', 'a-0011']
payment-svc|db_connection_pool_used_ratio|warn              1  ['a-0001']
payment-svc|error_rate|warn                                 1  ['a-0004']
checkout-svc|latency_p99_ms|warn                            1  ['a-0005']
checkout-svc|downstream_payment_error_rate|crit             1  ['a-0006']
edge-lb|upstream_5xx_rate|warn                              1  ['a-0007']
cart-svc|latency_p99_ms|warn                                1  ['a-0009']
notification-svc|queue_lag_ms|warn                          1  ['a-0010']
checkout-svc|request_drop_rate|crit            

In [ ]:
# Layer 2 - Session Window
GAP_SEC = 120

sessions = session_groups(alerts, gap_sec=GAP_SEC)

print(f"Layer 2 - Session Window (gap_sec={GAP_SEC})")
print("=" * 60)
print(f"Sessions found: {len(sessions)}")
print()

for i, session in enumerate(sessions):
    t_min = min(a['ts'] for a in session)
    t_max = max(a['ts'] for a in session)
    services = sorted(set(a['service'] for a in session))
    print(f"Session {i}: {len(session)} alerts | {t_min} -> {t_max}")
    print(f"  Services: {services}")
    for a in session:
        print(f"    {a['id']}  {a['ts']}  {a['service']:20s} {a['metric']}")
    print()

Layer 2 - Session Window (gap_sec=120)
Sessions found: 1

Session 0: 20 alerts | 2026-06-12T09:42:01Z -> 2026-06-12T09:48:30Z
  Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
    a-0001  2026-06-12T09:42:01Z  payment-svc          db_connection_pool_used_ratio
    a-0002  2026-06-12T09:42:18Z  payment-svc          db_connection_pool_used_ratio
    a-0003  2026-06-12T09:42:22Z  payment-svc          latency_p99_ms
    a-0004  2026-06-12T09:42:30Z  payment-svc          error_rate
    a-0005  2026-06-12T09:42:45Z  checkout-svc         latency_p99_ms
    a-0006  2026-06-12T09:43:01Z  checkout-svc         downstream_payment_error_rate
    a-0007  2026-06-12T09:43:15Z  edge-lb              upstream_5xx_rate
    a-0008  2026-06-12T09:43:18Z  payment-svc          latency_p99_ms
    a-0009  2026-06-12T09:43:32Z  cart-svc             latency_p99_ms
    a-0010  2026-06-12T09:43:50Z  notification-svc     queue_lag_ms
    a-0011  

In [ ]:
# Topology Analysis
MAX_HOP = 2

print(f"Layer 3 - Topology Analysis (max_hop={MAX_HOP})")
print("=" * 60)

alerting_services = sorted(set(a['service'] for a in alerts))
print(f"Services with alerts: {alerting_services}")
print()

undirected = graph.to_undirected()
print(f"{'Service A':20s} {'Service B':20s} {'Distance':>8}  Within max_hop?")
print(f"{'-'*20} {'-'*20} {'-'*8}  {'-'*15}")

for i, s1 in enumerate(alerting_services):
    for s2 in alerting_services[i+1:]:
        try:
            dist = nx.shortest_path_length(undirected, s1, s2)
            within = 'YES' if dist <= MAX_HOP else 'NO'
            print(f"{s1:20s} {s2:20s} {dist:8d}  {within}")
        except nx.NetworkXNoPath:
            print(f"{s1:20s} {s2:20s} {'inf':>8s}  NO PATH")

Layer 3 - Topology Analysis (max_hop=2)
Services with alerts: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']

Service A            Service B            Distance  Within max_hop?
-------------------- -------------------- --------  ---------------
cart-svc             checkout-svc                1  YES
cart-svc             edge-lb                     2  YES
cart-svc             notification-svc            2  YES
cart-svc             payment-svc                 2  YES
cart-svc             recommender-svc             2  YES
cart-svc             search-svc                  3  NO
checkout-svc         edge-lb                     1  YES
checkout-svc         notification-svc            1  YES
checkout-svc         payment-svc                 1  YES
checkout-svc         recommender-svc             3  NO
checkout-svc         search-svc                  2  YES
edge-lb              notification-svc            2  YES
edge-lb              pa

In [ ]:
# Full Pipeline
MAX_HOP = 2

clusters = correlate(alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)
summary  = build_summary(alerts, clusters)

print(f"Full Pipeline Results (gap_sec={GAP_SEC}, max_hop={MAX_HOP})")
print("=" * 60)
print(f"Input alerts:    {summary['input_alerts']}")
print(f"Output clusters: {summary['output_clusters']}")
print(f"Reduction ratio: {summary['reduction_ratio']:.1%}")
print()

for c in clusters:
    print(f"--- Cluster {c['cluster_id']} ---")
    print(f"  Alert count:   {c['alert_count']}")
    print(f"  Services:      {c['services']}")
    print(f"  Time range:    {c['time_range'][0]} -> {c['time_range'][1]}")
    print(f"  Max severity:  {c['max_severity']}")
    print(f"  Fingerprints:  {c['fingerprints']}")
    print(f"  Alert IDs:     {c['alert_ids']}")
    print()

ratio = summary['reduction_ratio']
print("=" * 60)
print("ACCEPTANCE CHECK:")
print(f"  reduction_ratio = {ratio:.2f} {'PASS (>= 0.5)' if ratio >= 0.5 else 'FAIL (< 0.5)'}")

Full Pipeline Results (gap_sec=120, max_hop=2)
Input alerts:    20
Output clusters: 1
Reduction ratio: 95.0%

--- Cluster c-000-000 ---
  Alert count:   20
  Services:      ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
  Time range:    2026-06-12T09:42:01Z -> 2026-06-12T09:48:30Z
  Max severity:  crit
  Fingerprints:  ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|error_rate|crit', 'payment-svc|error_rate|warn', 'payment-svc|latency_p99_ms|crit', 'recommender-svc|cpu_utilization|warn', 'search-svc|

In [22]:
# Cell 6: Write Output
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

output_path = results_dir / "cluster_summary.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Written to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")
print()

with open(output_path, "r", encoding="utf-8") as f:
    validated = json.load(f)
print(f"Valid JSON - {validated['output_clusters']} clusters, reduction_ratio={validated['reduction_ratio']}")
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))

Written to results\cluster_summary.json
File size: 1751 bytes

Valid JSON - 1 clusters, reduction_ratio=0.95

{
  "input_alerts": 20,
  "output_clusters": 1,
  "reduction_ratio": 0.95,
  "clusters": [
    {
      "cluster_id": "c-000-000",
      "alert_count": 20,
      "services": [
        "cart-svc",
        "checkout-svc",
        "edge-lb",
        "notification-svc",
        "payment-svc",
        "recommender-svc",
        "search-svc"
      ],
      "alert_ids": [
        "a-0001",
        "a-0002",
        "a-0003",
        "a-0004",
        "a-0008",
        "a-0011",
        "a-0015",
        "a-0018",
        "a-0005",
        "a-0006",
        "a-0012",
        "a-0017",
        "a-0007",
        "a-0014",
        "a-0020",
        "a-0009",
        "a-0010",
        "a-0019",
        "a-0013",
        "a-0016"
      ],
      "fingerprints": [
        "cart-svc|latency_p99_ms|warn",
        "checkout-svc|downstream_payment_error_rate|crit",
        "checkout-svc|latency_p9